# Implementing `Scaling Laws for Neural Language Models` By `(Kaplan et al., 2020)`

### 1. Setup and Imports

In [9]:
import os
import math
import time

In [10]:
import warnings
import matplotlib
import numpy as np
import torch.nn as nn
import urllib.request
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from torch.utils.data import Dataset, DataLoader

matplotlib.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'lines.linewidth': 2,
    'figure.dpi': 100,
})

warnings.filterwarnings('ignore')

In [3]:
import torch

In [6]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

Using device: mps
PyTorch version: 2.8.0


In [7]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

### 2. Dataset Preparation

We use **Tiny Shakespeare** — a small (~1MB) text corpus of all Shakespeare works. Since our goal is to show scaling paper we use **character-level tokenization** for simplicity (each character = one token).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__))
                         if '__file__' in dir() else '.', 'shakespeare.txt')# 

In [12]:
if not os.path.exists(DATA_PATH):
    print("Downloading Tiny Shakespeare dataset...")
    try:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
        print(f"Downloaded to {DATA_PATH}")
    except Exception:
        # Fallback generating synthetic text if download fails
        print("Download failed. Generating synthetic text corpus...")
        np.random.seed(SEED)
        chars = list("abcdefghijklmnopqrstuvwxyz .,:;!?\n'")
        # Create text with some structure (repeated patterns help scaling laws show)
        phrases = [
            "to be or not to be that is the question ",
            "all the world is a stage and all the men and women merely players ",
            "the lady doth protest too much methinks ",
            "brevity is the soul of wit ",
            "though this be madness yet there is method in it ",
            "something is rotten in the state of denmark ",
            "we know what we are but know not what we may be ",
            "love all trust a few do wrong to none ",
            "the course of true love never did run smooth ",
            "all that glitters is not gold ",
        ]
        text_chunks = []
        for _ in range(5000):
            text_chunks.append(np.random.choice(phrases))
        synthetic_text = "".join(text_chunks)
        with open(DATA_PATH, 'w') as f:
            f.write(synthetic_text)
        print(f"Generated {len(synthetic_text)} characters of synthetic text.")

Downloaded to ./shakespeare.txt


In [13]:
with open(DATA_PATH, 'r') as f:
    raw_text = f.read()

In [14]:
print(f"Dataset size: {len(raw_text):,} characters")
print(f"Sample: {raw_text[:200]}")

Dataset size: 1,115,394 characters
Sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you
